In [ ]:


# 读取数据
df = pd.read_csv('VEHICLES.csv')

# 数据清洗
# 重命名第一列
df = df.rename(columns={'': 'ID'})

# 检查数据类型
print("数据类型检查:")
print(df.dtypes)

# 基本统计信息
print("\n基本统计信息:")
print(df.describe())

# 检查缺失值
print("\n缺失值检查:")
print(df.isnull().sum())

# 数据可视化

# 1. 车辆性能雷达图 - 选取几个有代表性的车辆进行比较
def create_radar_chart(df, vehicles):
    # 选择要显示的属性
    attributes = ['GroundSpeed', 'WaterSpeed', 'AirSpeed', 'AntiGravitySpeed', 'Acceleration', 
                 'Weight', 'GroundHandling', 'WaterHandling', 'AirHandling', 
                 'AntiGravityHandling', 'Traction', 'MiniTurbo']
    
    # 创建雷达图
    fig = go.Figure()
    
    for vehicle in vehicles:
        vehicle_data = df[df['Vehicle'] == vehicle]
        values = vehicle_data[attributes].values.flatten().tolist()
        # 添加第一个值以闭合雷达图
        values.append(values[0])
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=attributes + [attributes[0]],
            fill='toself',
            name=vehicle
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[-1, 1]
            )
        ),
        title="车辆性能雷达图对比",
        showlegend=True
    )
    
    return fig

# 选择几个有代表性的车辆
selected_vehicles = ['Standard Kart', 'Blue Falcon', 'Cat Cruiser', 'Badwagon', 'Sport Bike+']
radar_fig = create_radar_chart(df, selected_vehicles)
radar_fig.show()

# 2. 速度与操控性散点图
def create_speed_handling_scatter(df):
    # 计算平均速度和平均操控性
    df['AvgSpeed'] = df[['GroundSpeed', 'WaterSpeed', 'AirSpeed', 'AntiGravitySpeed']].mean(axis=1)
    df['AvgHandling'] = df[['GroundHandling', 'WaterHandling', 'AirHandling', 'AntiGravityHandling']].mean(axis=1)
    
    # 创建散点图
    fig = px.scatter(
        df, 
        x='AvgSpeed', 
        y='AvgHandling', 
        text='Vehicle',
        size='Weight',
        color='Acceleration',
        color_continuous_scale='RdBu',
        hover_data=['GroundSpeed', 'Acceleration', 'MiniTurbo'],
        title='车辆速度与操控性对比'
    )
    
    fig.update_traces(
        textposition='top center',
        marker=dict(size=10, opacity=0.8),
        selector=dict(mode='markers+text')
    )
    
    fig.update_layout(
        xaxis_title='平均速度',
        yaxis_title='平均操控性',
        coloraxis_colorbar=dict(title='加速度')
    )
    
    return fig

speed_handling_fig = create_speed_handling_scatter(df)
speed_handling_fig.show()

# 3. 各属性分布柱状图
def create_attribute_distribution(df):
    # 计算每个车辆在各个属性上的排名
    attributes = ['GroundSpeed', 'WaterSpeed', 'AirSpeed', 'AntiGravitySpeed', 
                 'Acceleration', 'Weight', 'Traction', 'MiniTurbo']
    
    # 创建子图
    fig = make_subplots(rows=2, cols=4, subplot_titles=attributes)
    
    # 添加每个属性的直方图
    for i, attr in enumerate(attributes):
        row = i // 4 + 1
        col = i % 4 + 1
        
        fig.add_trace(
            go.Histogram(
                x=df[attr],
                nbinsx=10,
                marker_color='royalblue',
                opacity=0.75
            ),
            row=row, col=col
        )
        
        fig.update_xaxes(title_text=attr, row=row, col=col)
        fig.update_yaxes(title_text="数量", row=row, col=col)
    
    fig.update_layout(
        title_text="车辆属性分布",
        showlegend=False,
        height=600,
        width=1000
    )
    
    return fig

attribute_dist_fig = create_attribute_distribution(df)
attribute_dist_fig.show()

# 4. 相关性热图
def create_correlation_heatmap(df):
    # 选择数值型列计算相关性
    numeric_cols = ['GroundSpeed', 'WaterSpeed', 'AirSpeed', 'AntiGravitySpeed', 
                   'Acceleration', 'Weight', 'GroundHandling', 'WaterHandling', 
                   'AirHandling', 'AntiGravityHandling', 'Traction', 'MiniTurbo']
    
    corr = df[numeric_cols].corr()
    
    # 创建热图
    fig = go.Figure(data=go.Heatmap(
        z=corr.values,
        x=corr.columns,
        y=corr.index,
        colorscale='RdBu_r',
        zmid=0,
        colorbar=dict(title='相关系数')
    ))
    
    fig.update_layout(
        title='车辆属性相关性热图',
        xaxis_title='属性',
        yaxis_title='属性',
        width=800,
        height=700
    )
    
    return fig

corr_heatmap_fig = create_correlation_heatmap(df)
corr_heatmap_fig.show()

# 5. 车辆性能综合分析
def create_performance_score(df):
    # 创建不同场景下的性能分数
    # 1. 速度型赛道 (重视速度和加速度)
    df['SpeedScore'] = (df['GroundSpeed'] * 2 + df['Acceleration'] * 1.5 + 
                        df['AntiGravitySpeed'] * 1 + df['MiniTurbo'] * 0.5)
    
    # 2. 技术型赛道 (重视操控性和抓地力)
    df['TechnicalScore'] = (df['GroundHandling'] * 2 + df['Traction'] * 1.5 + 
                           df['MiniTurbo'] * 1 + df['Acceleration'] * 0.5)
    
    # 3. 水上赛道 (重视水上性能)
    df['WaterScore'] = (df['WaterSpeed'] * 2 + df['WaterHandling'] * 2 + 
                        df['Acceleration'] * 0.5 + df['MiniTurbo'] * 0.5)
    
    # 4. 空中赛道 (重视空中性能)
    df['AirScore'] = (df['AirSpeed'] * 2 + df['AirHandling'] * 2 + 
                      df['Acceleration'] * 0.5 + df['MiniTurbo'] * 0.5)
    
    # 创建子图
    fig = make_subplots(rows=2, cols=2, 
                        subplot_titles=('速度型赛道推荐', '技术型赛道推荐', 
                                        '水上赛道推荐', '空中赛道推荐'))
    
    # 添加每种场景的前10车辆
    scenarios = [('SpeedScore', 1, 1), ('TechnicalScore', 1, 2), 
                ('WaterScore', 2, 1), ('AirScore', 2, 2)]
    
    colors = px.colors.qualitative.Set3
    
    for i, (score, row, col) in enumerate(scenarios):
        # 获取前10名车辆
        top10 = df.sort_values(by=score, ascending=False).head(10)
        
        fig.add_trace(
            go.Bar(
                x=top10['Vehicle'],
                y=top10[score],
                marker_color=colors[i],
                name=score.replace('Score', '')
            ),
            row=row, col=col
        )
        
        fig.update_xaxes(title_text='车辆', tickangle=45, row=row, col=col)
        fig.update_yaxes(title_text='得分', row=row, col=col)
    
    fig.update_layout(
        title_text="不同赛道类型的车辆推荐",
        showlegend=True,
        height=800,
        width=1000
    )
    
    return fig, df

performance_fig, enhanced_df = create_performance_score(df)
performance_fig.show()

# 显示增强后的数据框中的各种性能得分
print("\n车辆综合性能得分:")
print(enhanced_df[['Vehicle', 'SpeedScore', 'TechnicalScore', 'WaterScore', 'AirScore']].head(10))

# 保存清洗和增强后的数据
enhanced_df.to_csv('VEHICLES_enhanced.csv', index=False)

print("\n数据处理和可视化完成，增强数据已保存至 'VEHICLES_enhanced.csv'")

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 读取数据
df = pd.read_csv('VEHICLES.csv')

# 数据清洗
# 重命名第一列
df = df.rename(columns={'': 'ID'})

# 检查数据类型
print("数据类型检查:")
print(df.dtypes)

# 基本统计信息
print("\n基本统计信息:")
print(df.describe())

# 检查缺失值
print("\n缺失值检查:")
print(df.isnull().sum())

# 数据可视化

# 1. 车辆性能雷达图 - 选取几个有代表性的车辆进行比较
def create_radar_chart(df, vehicles):
    # 选择要显示的属性
    attributes = ['GroundSpeed', 'WaterSpeed', 'AirSpeed', 'AntiGravitySpeed', 'Acceleration', 
                 'Weight', 'GroundHandling', 'WaterHandling', 'AirHandling', 
                 'AntiGravityHandling', 'Traction', 'MiniTurbo']
    
    # 创建雷达图
    fig = go.Figure()
    
    for vehicle in vehicles:
        vehicle_data = df[df['Vehicle'] == vehicle]
        values = vehicle_data[attributes].values.flatten().tolist()
        # 添加第一个值以闭合雷达图
        values.append(values[0])
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=attributes + [attributes[0]],
            fill='toself',
            name=vehicle
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[-1, 1]
            )
        ),
        title="车辆性能雷达图对比",
        showlegend=True
    )
    
    return fig

# 选择几个有代表性的车辆
selected_vehicles = ['Standard Kart', 'Blue Falcon', 'Cat Cruiser', 'Badwagon', 'Sport Bike+']
radar_fig = create_radar_chart(df, selected_vehicles)
radar_fig.show()

# 2. 速度与操控性散点图
def create_speed_handling_scatter(df):
    # 计算平均速度和平均操控性
    df['AvgSpeed'] = df[['GroundSpeed', 'WaterSpeed', 'AirSpeed', 'AntiGravitySpeed']].mean(axis=1)
    df['AvgHandling'] = df[['GroundHandling', 'WaterHandling', 'AirHandling', 'AntiGravityHandling']].mean(axis=1)
    
    # 创建散点图
    fig = px.scatter(
        df, 
        x='AvgSpeed', 
        y='AvgHandling', 
        text='Vehicle',
        size='Weight',
        color='Acceleration',
        color_continuous_scale='RdBu',
        hover_data=['GroundSpeed', 'Acceleration', 'MiniTurbo'],
        title='车辆速度与操控性对比'
    )
    
    fig.update_traces(
        textposition='top center',
        marker=dict(size=10, opacity=0.8),
        selector=dict(mode='markers+text')
    )
    
    fig.update_layout(
        xaxis_title='平均速度',
        yaxis_title='平均操控性',
        coloraxis_colorbar=dict(title='加速度')
    )
    
    return fig

speed_handling_fig = create_speed_handling_scatter(df)
speed_handling_fig.show()

# 3. 各属性分布柱状图
def create_attribute_distribution(df):
    # 计算每个车辆在各个属性上的排名
    attributes = ['GroundSpeed', 'WaterSpeed', 'AirSpeed', 'AntiGravitySpeed', 
                 'Acceleration', 'Weight', 'Traction', 'MiniTurbo']
    
    # 创建子图
    fig = make_subplots(rows=2, cols=4, subplot_titles=attributes)
    
    # 添加每个属性的直方图
    for i, attr in enumerate(attributes):
        row = i // 4 + 1
        col = i % 4 + 1
        
        fig.add_trace(
            go.Histogram(
                x=df[attr],
                nbinsx=10,
                marker_color='royalblue',
                opacity=0.75
            ),
            row=row, col=col
        )
        
        fig.update_xaxes(title_text=attr, row=row, col=col)
        fig.update_yaxes(title_text="数量", row=row, col=col)
    
    fig.update_layout(
        title_text="车辆属性分布",
        showlegend=False,
        height=600,
        width=1000
    )
    
    return fig

attribute_dist_fig = create_attribute_distribution(df)
attribute_dist_fig.show()

# 4. 相关性热图
def create_correlation_heatmap(df):
    # 选择数值型列计算相关性
    numeric_cols = ['GroundSpeed', 'WaterSpeed', 'AirSpeed', 'AntiGravitySpeed', 
                   'Acceleration', 'Weight', 'GroundHandling', 'WaterHandling', 
                   'AirHandling', 'AntiGravityHandling', 'Traction', 'MiniTurbo']
    
    corr = df[numeric_cols].corr()
    
    # 创建热图
    fig = go.Figure(data=go.Heatmap(
        z=corr.values,
        x=corr.columns,
        y=corr.index,
        colorscale='RdBu_r',
        zmid=0,
        colorbar=dict(title='相关系数')
    ))
    
    fig.update_layout(
        title='车辆属性相关性热图',
        xaxis_title='属性',
        yaxis_title='属性',
        width=800,
        height=700
    )
    
    return fig

corr_heatmap_fig = create_correlation_heatmap(df)
corr_heatmap_fig.show()

# 5. 车辆性能综合分析
def create_performance_score(df):
    # 创建不同场景下的性能分数
    # 1. 速度型赛道 (重视速度和加速度)
    df['SpeedScore'] = (df['GroundSpeed'] * 2 + df['Acceleration'] * 1.5 + 
                        df['AntiGravitySpeed'] * 1 + df['MiniTurbo'] * 0.5)
    
    # 2. 技术型赛道 (重视操控性和抓地力)
    df['TechnicalScore'] = (df['GroundHandling'] * 2 + df['Traction'] * 1.5 + 
                           df['MiniTurbo'] * 1 + df['Acceleration'] * 0.5)
    
    # 3. 水上赛道 (重视水上性能)
    df['WaterScore'] = (df['WaterSpeed'] * 2 + df['WaterHandling'] * 2 + 
                        df['Acceleration'] * 0.5 + df['MiniTurbo'] * 0.5)
    
    # 4. 空中赛道 (重视空中性能)
    df['AirScore'] = (df['AirSpeed'] * 2 + df['AirHandling'] * 2 + 
                      df['Acceleration'] * 0.5 + df['MiniTurbo'] * 0.5)
    
    # 创建子图
    fig = make_subplots(rows=2, cols=2, 
                        subplot_titles=('速度型赛道推荐', '技术型赛道推荐', 
                                        '水上赛道推荐', '空中赛道推荐'))
    
    # 添加每种场景的前10车辆
    scenarios = [('SpeedScore', 1, 1), ('TechnicalScore', 1, 2), 
                ('WaterScore', 2, 1), ('AirScore', 2, 2)]
    
    colors = px.colors.qualitative.Set3
    
    for i, (score, row, col) in enumerate(scenarios):
        # 获取前10名车辆
        top10 = df.sort_values(by=score, ascending=False).head(10)
        
        fig.add_trace(
            go.Bar(
                x=top10['Vehicle'],
                y=top10[score],
                marker_color=colors[i],
                name=score.replace('Score', '')
            ),
            row=row, col=col
        )
        
        fig.update_xaxes(title_text='车辆', tickangle=45, row=row, col=col)
        fig.update_yaxes(title_text='得分', row=row, col=col)
    
    fig.update_layout(
        title_text="不同赛道类型的车辆推荐",
        showlegend=True,
        height=800,
        width=1000
    )
    
    return fig, df

performance_fig, enhanced_df = create_performance_score(df)
performance_fig.show()

# 显示增强后的数据框中的各种性能得分
print("\n车辆综合性能得分:")
print(enhanced_df[['Vehicle', 'SpeedScore', 'TechnicalScore', 'WaterScore', 'AirScore']].head(10))

# 保存清洗和增强后的数据
enhanced_df.to_csv('VEHICLES_enhanced.csv', index=False)

print("\n数据处理和可视化完成，增强数据已保存至 'VEHICLES_enhanced.csv'")